In [2]:
import pandas as pd
import json

with open("outputs/activation-matching-results-mlp.json", "r") as f:
    data = json.load(f)
df = pd.DataFrame(
    [
        {
            "run_key": d["run_key"],
            "model1_index": d["model1_index"],
            "model2_index": d["model2_index"],
            "epoch": d["epoch"],
            "layer": r["layer"],
            "objective_optimal": r["objectives"]["optimal"],
            "objective_identity": r["objectives"]["identity"],
            "objective_random_mean": r["objectives"]["random_mean"],
            "objective_random_std": r["objectives"]["random_std"],
        }
        for d in data
        for r in d["matching_results"]
])

print("Available layers:", df["layer"].unique())
df = df[df["layer"].str.contains("input")]
print("Selected layers:", df["layer"].unique())

Available layers: ['lins.1.input' 'lins.2.input' 'lins.3.input' 'lins.0.output'
 'lins.1.output' 'lins.2.output']
Selected layers: ['lins.1.input' 'lins.2.input' 'lins.3.input']


In [3]:
from IPython.display import display
import ipywidgets as widgets
import plotly.graph_objects as go

def plot_grid(figures, width=500, height=300):
    """
    Display figures in a grid with complete separation.
    """
    rows = []
    
    for row_figs in figures:
        fig_widgets = []
        for fig in row_figs:
            if fig is None:
                continue
            
            # Create FigureWidget from existing figure's data and layout
            fw = go.FigureWidget(data=fig.data, layout=fig.layout)
            fw.update_layout(width=width, height=height)
            fig_widgets.append(fw)
        
        rows.append(widgets.HBox(fig_widgets))
    
    grid = widgets.VBox(rows)
    display(grid)

In [4]:
import plotly.graph_objects as go

def plot_objectives_over_epochs(df, run_key, layer_name, log_y_axis=True, show_min_max=False):
    """
    For a single run_key:
    - Average over (model1_index, model2_index) per epoch
    - Plot mean ± std for each objective
    """
    d = df[df["run_key"] == run_key]

    # Aggregate per epoch
    agg = (
        d.groupby("epoch")
        .agg(
            optimal_mean=("objective_optimal", "mean"),
            optimal_min=("objective_optimal", "min"),
            optimal_max=("objective_optimal", "max"),
            optimal_std=("objective_optimal", "std"),
            identity_mean=("objective_identity", "mean"),
            identity_min=("objective_identity", "min"),
            identity_max=("objective_identity", "max"),
            identity_std=("objective_identity", "std"),
            random_mean=("objective_random_mean", "mean"),
            random_min=("objective_random_mean", "min"),
            random_max=("objective_random_mean", "max"),
            random_std=("objective_random_mean", "std"),
        )
        .reset_index()
        .sort_values("epoch")
    )
    # print(agg)

    fig = go.Figure()

    def add_mean_min_max(x, mean, min, max, name, color, dash, line_width):
        fig.add_trace(go.Scatter(
            x=x,
            y=mean,
            mode="lines",
            name=name,
            line=dict(color=color, dash=dash, width=line_width),
        ))

        if show_min_max:
            # fig.add_trace(go.Scatter(
            #     x=list(x) + list(x[::-1]),
            #     y=list(max) + list(min)[::-1],
            #     fill="toself",
            #     fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.2)"),
            #     line=dict(color="rgba(255,255,255,0)"),
            #     hoverinfo="skip",
            #     showlegend=False,
            # ))
            fig.add_trace(go.Scatter(
                x=list(x),
                y=list(max),
                line=dict(color=color[:-1].replace("rgb", "rgba") + ",10)", width=0.5),
                hoverinfo="skip",
                showlegend=False,
            ))
            fig.add_trace(go.Scatter(
                x=list(x),
                y=list(min),
                line=dict(color=color[:-1].replace("rgb", "rgba") + ",10)", width=0.5),
                hoverinfo="skip",
                showlegend=False,
            ))

    add_mean_min_max(
        agg["epoch"],
        agg["optimal_mean"],
        agg["optimal_min"],
        agg["optimal_max"],
        "Optimal",
        "rgb(31, 119, 180)",
        "solid", 1
    )

    add_mean_min_max(
        agg["epoch"],
        agg["identity_mean"],
        agg["identity_min"],
        agg["identity_max"],
        "Identity",
        "rgb(255, 127, 14)",
        "dash", 3
    )

    add_mean_min_max(
        agg["epoch"],
        agg["random_mean"],
        agg["random_min"],
        agg["random_max"],
        "Random",
        "rgb(44, 160, 44)",
        "dot", 2
    )
    
    fig.add_trace(go.Scatter(
        x=list(agg["epoch"]) + list(agg["epoch"])[::-1],
        y=list(agg["random_mean"]+agg["random_std"]) + list(agg["random_mean"]-agg["random_std"])[::-1],
        line=dict(color="rgba(44, 160, 44, 20)", width=0.5),
        fill="toself",
        hoverinfo="skip",
        showlegend=False,
    ))

    fig.update_layout(
        title=f"{run_key} / {layer_name}: Matching, averaged over 8 model pairs",
        xaxis_title="Epoch",
        yaxis_title="Activation match. objective",
        template="plotly_white",
        legend_title="Objective",
        height=300,
        margin=dict(l=0, r=0, t=70, b=0),
    )
    if log_y_axis:
        fig.update_layout(yaxis_type="log")

    return fig


# for run_key in df["run_key"].unique():
figures = [
    [
        plot_objectives_over_epochs(df[df["layer"] == layer_name], run_key=run_key, layer_name=layer_name, log_y_axis=False, show_min_max=False)
        for layer_name in sorted(df["layer"].unique().tolist())
    ]
    for run_key in ["mlp_symmetry0", "mlp_symmetry1_kappa0", "mlp_symmetry1_kappa1", "mlp_symmetry2", "mlp_symmetry3_kappa1"]
]
plot_grid(figures, width=550, height=300)
# for run_key in ["mlp_symmetry0", "mlp_symmetry1_kappa0", "mlp_symmetry1_kappa1", "mlp_symmetry2", "mlp_symmetry3_kappa1"]:
#     for layer_name in df["layer"].unique():
#         fig = plot_objectives_over_epochs(df, run_key=run_key, log_y_axis=True, show_min_max=False)
#         fig.show()


    'data': [{'line': {'color': 'rgb(31, 119, 180)', 'dash': 'sol…

In [ ]:
def plot_objectives_at_epoch(df, epoch=100, plot_difference_from_random=False):
    """
    At a fixed epoch:
    - Average over (model1_index, model2_index) per run_key
    - Plot mean with std error bars
    """
    d = df[df["epoch"] == epoch]

    agg = (
        d.groupby("run_key")
        .agg(
            optimal_mean=("objective_optimal", "mean"),
            optimal_min=("objective_optimal", "min"),
            optimal_max=("objective_optimal", "max"),
            optimal_std=("objective_optimal", "std"),
            identity_mean=("objective_identity", "mean"),
            identity_min=("objective_identity", "min"),
            identity_max=("objective_identity", "max"),
            identity_std=("objective_identity", "std"),
            random_mean=("objective_random_mean", "mean"),
            random_min=("objective_random_mean", "min"),
            random_max=("objective_random_mean", "max"),
            random_std=("objective_random_mean", "std"),
        )
        .reset_index()
    )

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=agg["run_key"],
        y=agg["optimal_mean"]-agg["random_mean"]*float(plot_difference_from_random),
        mode="markers+lines",
        name="Optimal",
        line={"color": "rgb(31, 119, 180)", "dash": "solid"},
        # error_y=dict(type="data", array=agg["optimal_std"]),
        # error_y=dict(type="data", symmetric=False, array=(agg["optimal_max"]-agg["optimal_mean"]), arrayminus=(agg["optimal_mean"]-agg["optimal_min"])),
    ))

    fig.add_trace(go.Scatter(
        x=agg["run_key"],
        y=agg["identity_mean"]-agg["random_mean"]*float(plot_difference_from_random),
        mode="markers+lines",
        name="Identity",
        line={"color": "rgb(255, 127, 14)", "dash": "dash", "width": 3},
        # error_y=dict(type="data", array=agg["identity_std"]),
        # error_y=dict(type="data", symmetric=False, array=(agg["identity_max"]-agg["identity_mean"]), arrayminus=(agg["identity_mean"]-agg["identity_min"])),
    ))

    fig.add_trace(go.Scatter(
        x=agg["run_key"],
        y=agg["random_mean"]-agg["random_mean"]*float(plot_difference_from_random),
        mode="markers+lines",
        name="Random",
        line={"color": "rgb(44, 160, 44)", "dash": "dot", "width": 2},
        # error_y=dict(type="data", array=agg["random_std"]),
        # error_y=dict(type="data", symmetric=False, array=(agg["random_max"]-agg["random_mean"]), arrayminus=(agg["random_mean"]-agg["random_min"])),
    ))

    fig.update_layout(
        title=f"Activation matching objectives at Epoch {epoch}, averaged over 8 model pairs{' (difference from random)' if plot_difference_from_random else ''}",
        xaxis_title="Run Key",
        yaxis_title="Objective",
        template="plotly_white",
        legend_title="Objective",
        height=400
    )

    return fig


# plot_objectives_at_epoch(df[df["layer"] == "lins.2.input"], epoch=100, plot_difference_from_random=False).show()
plot_objectives_at_epoch(df[df["run_key"].isin(["mlp_symmetry0", "mlp_symmetry1_kappa0", "mlp_symmetry1_kappa1", "mlp_symmetry2", "mlp_symmetry3_kappa1"])], epoch=100, plot_difference_from_random=False).show()
plot_objectives_at_epoch(df, epoch=100, plot_difference_from_random=True).show()

In [9]:
df[df["run_key"] == "mlp_symmetry0"]

,run_key,model1_index,model2_index,epoch,layer,objective_optimal,objective_identity,objective_random_mean,objective_random_std
0,mlp_symmetry0,1,2,0,lins.1.input,0.164169,0.079509,0.080332,0.003481
1,mlp_symmetry0,1,2,0,lins.2.input,0.190085,0.081574,0.079019,0.004761
2,mlp_symmetry0,1,2,0,lins.3.input,0.216838,0.086011,0.079787,0.006606
6,mlp_symmetry0,1,2,5,lins.1.input,0.041220,0.024531,0.024499,0.000645
7,mlp_symmetry0,1,2,5,lins.2.input,0.068519,0.045382,0.045713,0.000961
...,...,...,...,...,...,...,...,...,...
997,mlp_symmetry0,15,16,95,lins.2.input,0.008173,0.000668,0.000739,0.000285
998,mlp_symmetry0,15,16,95,lins.3.input,0.018141,0.003929,0.003489,0.000597
1002,mlp_symmetry0,15,16,100,lins.1.input,0.003410,0.000142,0.000051,0.000143
1003,mlp_symmetry0,15,16,100,lins.2.input,0.008132,-0.000006,0.000471,0.000312


In [ ]:
# one-off experiment to measure L2 distances

import concurrent.futures
import hydra
import torch
import pathlib
import train
import src.utils.rebasin.activation_matching
import numpy as np
from contextlib import contextmanager
import tqdm

checkpoint_directories = {
    "mlp_symmetry0": "outputs/2025-12-17/12-18-16_mlp_mnist_sym-0__3pb1rw8b__kk13eg0q",
    "mlp_symmetry1_kappa0": "outputs/2025-12-17/14-20-51_mlp_mnist_sym-1__m3z8jvf9__iwzo2u22",
    "mlp_symmetry1_kappa1": "outputs/2025-12-17/12-30-55_mlp_mnist_sym-1__3pb1rw8b__4b0gt1xm",
    "mlp_symmetry1_kappaPerLayer": "outputs/2025-12-17/12-46-23_mlp_mnist_sym-1__3pb1rw8b__8fvvk8f3",
    "mlp_symmetry2": "outputs/2025-12-17/13-01-05_mlp_mnist_sym-2__3pb1rw8b__48q9fddu",
    "mlp_symmetry3_kappa0": "outputs/2025-12-17/13-16-26_mlp_mnist_sym-3__3pb1rw8b__tq3utpfq",
    "mlp_symmetry3_kappa1": "outputs/2025-12-17/13-30-38_mlp_mnist_sym-3__3pb1rw8b__ruzzxkpy",
    "mlp_symmetry3_kappaPerLayer": "outputs/2025-12-17/13-45-10_mlp_mnist_sym-3__3pb1rw8b__e4vv3n8v",
}

def measure_l2_distances(checkpoint_path_1, checkpoint_path_2, device="cuda:0", data_info=None):
    # epoch, model_1_index, model_2_index):
    # checkpoint_paths = [
    #     f"{checkpoint_directories[RUN_KEY]}/checkpoint_epoch_{epoch}_model_{model_1_index}.pt",
    #     f"{checkpoint_directories[RUN_KEY]}/checkpoint_epoch_{epoch}_model_{model_2_index}.pt"
    # ]

    with hydra.initialize(version_base=None, config_path=str(pathlib.Path(checkpoint_path_1).parent)):
        cfg = hydra.compose(config_name="config")
    cfg.dataset.batch_size = 10000
    if data_info is None:
        data_info = train.setup_data_loaders(cfg)
    
    param_vector_1 = torch.stack([v.reshape(-1) for _, v in torch.load(checkpoint_path_1, map_location=device)["model_state_dict"]])
    param_vector_2 = torch.stack([v.reshape(-1) for _, v in torch.load(checkpoint_path_2, map_location=device)["model_state_dict"]])
    return torch.linalg.norm(param_vector_1 - param_vector_2).item()


MODEL_1_RANGE = list(range(1, 17, 2))
EPOCH_RANGE = list(range(0, 101, 5))
MAX_PARALLEL_PROCESSES = 25
checkpoint_path = lambda model_index, epoch: f"{checkpoint_directories[run_key]}/checkpoint_epoch_{epoch}_model_{model_index}.pt"

all_l2_distances = []
for run_key in (tqdm_run := tqdm.tqdm(checkpoint_directories.keys())):
    tqdm_run.set_description(f"Run: {run_key}")
    for model1_index in (tqdm_model_pair := tqdm.tqdm(MODEL_1_RANGE, desc=run_key, leave=False)):
        model2_index = model1_index + 1
        tqdm_model_pair.set_description(f"Model pair: (Model {model1_index}, Model {model2_index})")
        with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_PARALLEL_PROCESSES) as executor:
            model_pair_results = [future.result() for future in [
                executor.submit(measure_l2_distances,
                    **dict(checkpoint_path_1=checkpoint_path(model1_index, epoch), checkpoint_path_2=checkpoint_path(model2_index, epoch))
                ) for epoch in EPOCH_RANGE]
            ]
            all_l2_distances.extend(({
                "run_key": run_key,
                "model1_index": model1_index,
                "model2_index": model2_index,
                "epoch": epoch,
                "distance": distance
                })
                for distance, epoch in zip(model_pair_results, EPOCH_RANGE)
            )

Run: mlp_symmetry0:   0%|          | 0/8 [00:19<?, ?it/s]


ValueError: too many values to unpack (expected 2)